In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [2]:
from ssd import SSD
from utils import normalised_gt_coords,corner_to_center,iou,center_to_corner,decode,encode ,matching



"""targets (tensor): Ground truth boxes and labels for a batch,
                shape: [batch_size,num_objs,5] (last idx is the label)."""

gt_boxes_batch = [
    # Image 1
    [
        (75, 80, 60, 90),
        (220, 70, 50, 40),
        (150, 170, 140, 110),
        (260, 240, 70, 120),
        (45, 230, 80, 60)
    ],

    # Image 2
    [
        (30, 40, 50, 60),
        (200, 120, 80, 90),
        (180, 160, 100, 110),
        (250, 210, 60, 80),
        (90, 200, 70, 50),
        (90, 200, 75, 50)
    ],
]

labels_list = [
    torch.tensor([1, 2, 1, 1, 2]),
    torch.tensor([1, 3, 3, 4, 2, 2]) 
]

# convert to tensors
gt_boxes_list = [torch.tensor(b, dtype=torch.float32) for b in gt_boxes_batch]

# gt=center_to_corner(normalised_gt_coords(gtboxes,300,300))


In [3]:
gt_list=[center_to_corner(normalised_gt_coords(gtboxes,300,300)) for gtboxes in gt_boxes_list]
gt_list

[tensor([[0.1500, 0.1167, 0.3500, 0.4167],
         [0.6500, 0.1667, 0.8167, 0.3000],
         [0.2667, 0.3833, 0.7333, 0.7500],
         [0.7500, 0.6000, 0.9833, 1.0000],
         [0.0167, 0.6667, 0.2833, 0.8667]]),
 tensor([[0.0167, 0.0333, 0.1833, 0.2333],
         [0.5333, 0.2500, 0.8000, 0.5500],
         [0.4333, 0.3500, 0.7667, 0.7167],
         [0.7333, 0.5667, 0.9333, 0.8333],
         [0.1833, 0.5833, 0.4167, 0.7500],
         [0.1750, 0.5833, 0.4250, 0.7500]])]

In [5]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [6]:
X=torch.randn((2,3,300,300))

model=SSD(base,21)

In [7]:
regressions,classifications,layers_for_prediction=model.forward(X)

In [11]:
"""#sa,ity chekc 
from utils import decode
decode(for_anchor_coords_encoded,model.anchors,[0.1,0.2])[312]


#shoud show tensor([0.1500, 0.1167, 0.3500, 0.4167])"""

'#sa,ity chekc \nfrom utils import decode\ndecode(for_anchor_coords_encoded,model.anchors,[0.1,0.2])[312]\n\n\n#shoud show tensor([0.1500, 0.1167, 0.3500, 0.4167])'

In [45]:
N_images=2
n_anchors=8732
coords_space=torch.empty((N_images,n_anchors,4))
labels_space=torch.empty((N_images,n_anchors,1))



for i in range(N_images):
    matching(model.anchors,gt_list[i],labels_list[i],coords_space,labels_space,i)

def smoothL1(x):
    return torch.where(torch.abs(x)<1,0.5*x**2,torch.abs(x)-0.5)
    
#lloc loss over positives only 

coords_space[(labels_space>0).squeeze()].shape

print(
smoothL1(regressions-coords_space)[(labels_space>0).squeeze()].sum()/(4)
)
nb_pos=len(regressions[(labels_space>0).squeeze()])
print(
F.smooth_l1_loss(regressions[(labels_space>0).squeeze()],coords_space[(labels_space>0).squeeze()], reduction='sum').item()/(4*nb_pos)
)
smoothL1(regressions-coords_space)[(labels_space>0).squeeze()].shape










tensor(148.7865, grad_fn=<DivBackward0>)
0.2403659604862934


torch.Size([619, 4])

In [44]:
len(regressions[(labels_space>0).squeeze()])

619

In [20]:
labels_space.shape

torch.Size([2, 8732, 1])

In [18]:
classifications.shape

torch.Size([2, 8732, 21])

In [24]:
classifications.reshape(2*8732,21)

tensor([[-5.1617e-02,  8.5901e-02,  4.1842e-02,  ..., -1.1395e-01,
          2.2417e-02,  1.2114e-02],
        [ 5.5833e-02,  7.1441e-02, -1.6426e-02,  ..., -2.0983e-03,
         -4.5428e-03, -8.1025e-02],
        [ 9.0354e-03, -3.9590e-02,  6.4773e-02,  ...,  1.9419e-02,
          3.6977e-02,  1.4256e-02],
        ...,
        [-1.2551e-02, -2.3277e-02,  4.8036e-03,  ..., -6.3132e-03,
          8.7506e-03,  4.5502e-04],
        [-1.5508e-02, -1.3934e-03,  1.1164e-04,  ...,  7.1126e-03,
         -1.0744e-02, -2.2420e-02],
        [-1.6860e-02,  6.9192e-03,  8.1439e-03,  ...,  1.3677e-02,
          1.4231e-02,  1.2901e-02]], grad_fn=<ViewBackward0>)

In [ ]:
labels_space.reshape(2*8732,).dt

tensor([0., 0., 0.,  ..., 0., 0., 0.])

In [46]:
F.cross_entropy(classifications.reshape(2*8732,21),labels_space.reshape(2*8732,).long(),reduction='sum')/(nb_pos)

tensor(42.9308, grad_fn=<DivBackward0>)

In [11]:
import torch
import torch.nn.functional as F

# Predicted logits for 3 samples and 4 classes
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.0],
    [0.5, 2.5, 0.3, 0.2],
    [0.1, 0.2, 3.0, 0.4]
])

# True class labels
targets = torch.tensor([0, 1, 2])

# Compute cross-entropy loss
loss = F.cross_entropy(logits, targets)

print("Cross Entropy Loss:", loss.item())

Cross Entropy Loss: 0.32464537024497986


In [450]:
regressions.shape

torch.Size([10, 8732, 4])

In [36]:
for_anchor_coords.shape

torch.Size([8732, 4])

# model forward pass 

In [44]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
regressions,classifications,layers_for_prediction=model.forward(X)

In [45]:
classifications.shape

torch.Size([10, 8732, 21])

In [46]:
regressions.shape



torch.Size([10, 8732, 4])

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
